# Step 4: Deploy the model and add data and model monitoring

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.6.0</strong> and the <strong>SageMaker Python SDK version 3.27.0</strong>.
</div>

In this step you deploy the model version that the pipeline from [step 3](03-sagemaker-pipeline.ipynb) registered, and you monitor it. You don't retrain the model. The notebook has two parts:
- **Part 1: Deploy**. Approve the model version in the [SageMaker Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html), deploy it to a [real-time endpoint](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html) with [data capture](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-data-capture.html), and send traffic to it.
- **Part 2: Monitor**. Detect data drift and evaluate the model quality on the captured data with the open-source library [Evidently](https://www.evidentlyai.com/evidently-oss), log the reports in MLflow, and run the checks in a scheduled SageMaker pipeline that sends alerts.

||||
|---|---|---|
|1. |Experiment in a notebook ||
|2. |Scale with SageMaker AI jobs and SageMaker SDK ||
|3. |Operationalize with ML pipeline, model registry ||
|4. |Deploy the model and add data and model monitoring |**<<<< YOU ARE HERE**|
|5. |Automate model deployment with a SageMaker project ||

## Workflow
The diagram shows the monitoring workflow with a real-time endpoint:

1. Take the approved model version from the SageMaker Model Registry and the baseline dataset that the pipeline stored in S3
2. Deploy the model to a real-time endpoint with data capture. Data capture stores the inputs and outputs of the endpoint in S3.
3. Calculate the data drift and the model quality with Evidently, from the baseline dataset and the captured data. You run the calculation in the notebook first, then in a [processing job](https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job.html) in a [pipeline](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines.html) on a schedule.
4. Log the data drift and model quality reports in MLflow, to track and compare them
5. Send an alert through [Amazon SNS](https://docs.aws.amazon.com/sns/latest/dg/welcome.html) if the drift or the model quality crosses a threshold

![Model monitoring workflow with real-time endpoints](img/arch-sagemaker-inference-predictiveml-monitoring-RealTime-inf.png "Model monitoring workflow with real-time endpoints")

<div class="alert alert-info">The diagram shows the general pattern. In this workshop the baseline comes from the preprocess step of the pipeline in step 3, not from the training job, and the alert is an SNS email instead of a Slack message.</div>

### Business problem
The model predicts whether a bank client subscribes to a term deposit after a marketing call. It uses the client's demographics (age, job, marital status, education), financial information (credit default, housing loan, personal loan), and campaign data (contact type, month, day of week, number of contacts, outcome of the previous campaign). It doesn't use the call duration or the economic indicators, because they aren't known before the call.

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), (
    f"This workshop needs Python 3.12, but the kernel runs Python {sys.version_info.major}.{sys.version_info.minor}. "
    "Run ./setup-py312.sh in a terminal, refresh the browser, and select the Python 3.12 kernel. "
    "See 00-start-here.ipynb for the steps."
)

---

## Setup
Load the stored variables and import the packages.

In [ ]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

In [ ]:
import boto3
import sagemaker
import mlflow
import pandas as pd
import numpy as np
import os
import json
import time
import importlib.util
from datetime import datetime
from sklearn.metrics import roc_auc_score
from IPython.display import HTML, display
from html import escape

from sagemaker.core.helper.session_helper import Session
from sagemaker.core.common_utils import name_from_base
from sagemaker.core.resources import Model, EndpointConfig, Endpoint, ModelPackage
from sagemaker.core.shapes import (
    ContainerDefinition, ProductionVariant,
    DataCaptureConfig, CaptureOption, CaptureContentTypeHeader,
)

from evidently import Report, Dataset, DataDefinition, BinaryClassification
from evidently.presets import DataDriftPreset, DataSummaryPreset, ClassificationPreset
from evidently.metrics import *

# Same feature engineering the pipeline's preprocess step uses
from pipeline_steps.preprocess import engineer_features

print(f'MLflow version: {mlflow.__version__}')

In [ ]:
sagemaker_session = Session()

print('AWS Configuration:')
print(f'  Region: {region}')
print(f'  S3 Bucket: {bucket_name}')
print(f'  IAM Role: {sm_role}')
print(f'  Data Prefix: {bucket_prefix}')

### Configure MLflow
The monitoring runs go to their own MLflow experiment, `from-idea-to-prod-monitoring`, so that they don't mix with the training runs of steps 1 to 3. `mlflow.set_experiment()` creates the experiment if it doesn't exist.

In [ ]:
# MLflow prints "View run ... at: https://mlflow.sagemaker.<region>.app.aws/..." when a run
# ends. That host is the SageMaker MLflow *tracking API* endpoint, not the UI, so the links
# do not open. Suppress them; the notebook shows working presigned UI links instead.
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "true"
mlflow.set_tracking_uri(mlflow_arn)
mlflow_experiment_name = 'from-idea-to-prod-monitoring'
monitoring_experiment = mlflow.set_experiment(mlflow_experiment_name)
print(f'MLflow experiment: {mlflow_experiment_name} (ID {monitoring_experiment.experiment_id})')

---

## Part 1: Deploy the approved model

### Load the datasets of the pipeline

The model you deploy was trained by the SageMaker pipeline in [notebook 03](03-sagemaker-pipeline.ipynb). To monitor it, you need data in exactly the format that model was trained on, so this notebook reuses the pipeline's outputs instead of preparing its own:

- `test/test_x.csv` and `test/test_y.csv`: the held-out test features and labels, which play the role of production traffic and ground truth
- `baseline/baseline.csv`: the full feature table, used as the reference distribution for drift detection

The pipeline writes these files without a header. The feature names come from `engineer_features()` in [`pipeline_steps/preprocess.py`](pipeline_steps/preprocess.py), the same function the pipeline's preprocess step runs.

In [ ]:
raw_df = pd.read_csv('data/bank-additional/bank-additional-full.csv', sep=';')
feature_names = engineer_features(raw_df).columns.drop('y').tolist()

X_test = pd.read_csv(f'{test_s3_url}/test_x.csv', header=None, names=feature_names)
y_test = pd.read_csv(f'{test_s3_url}/test_y.csv', header=None).iloc[:, 0].astype(int)
baseline_df = pd.read_csv(f'{baseline_s3_url}/baseline.csv', header=None, names=feature_names)

assert len(X_test) == len(y_test), 'test_x and test_y have different lengths'
print(f'Features:      {len(feature_names)}')
print(f'Test set:      {X_test.shape}  (positive rate {y_test.mean():.3f})')
print(f'Baseline set:  {baseline_df.shape}')
X_test.head()

Save the baseline **with a header** and upload it to S3. The Evidently reports and the monitoring processing job read the baseline by column name.

In [ ]:
os.makedirs('data', exist_ok=True)
baseline_df.to_csv('data/baseline.csv', index=False, header=True)
baseline_s3 = sagemaker_session.upload_data('data/baseline.csv', bucket_name, f'{bucket_prefix}/data/baseline')
print(f'Baseline S3: {baseline_s3}')

### Select the model version

The register step of the pipeline logged an **inference specification** on the MLflow model before it registered the model. The model package in the SageMaker Model Registry is therefore deployable as it is: it has the serving image, the location of the model data, and the environment variables.

The next cell selects the model version from the model package group of the pipeline:

1. the most recent version that is already `Approved` (for example by the CI/CD flow in [notebook 05](05-templatize-deployment.ipynb) on a re-run of the workshop), otherwise
2. the most recent `PendingManualApproval` version, which you approve in the next step.

The cell considers only versions with an `InferenceSpecification`. You can't deploy older versions without one from the registry.

In [ ]:
sm_client = boto3.client('sagemaker')

# Model package group written by the pipeline (notebook 03), restored via %store
try:
    model_package_group = target_model_package_group_name
except NameError:
    try:
        model_package_group = model_package_group_name
    except NameError:
        model_package_group = registered_model_name
print(f'Model package group: {model_package_group}')

def list_packages(status):
    return [
        p for page in sm_client.get_paginator('list_model_packages').paginate(
            ModelPackageGroupName=model_package_group,
            ModelApprovalStatus=status,
            SortBy='CreationTime',
            SortOrder='Descending',
        )
        for p in page['ModelPackageSummaryList']
    ]

def is_deployable(arn):
    return 'InferenceSpecification' in sm_client.describe_model_package(ModelPackageName=arn)

model_package_arn = next((p['ModelPackageArn'] for p in list_packages('Approved') if is_deployable(p['ModelPackageArn'])), None)
needs_approval = model_package_arn is None
if needs_approval:
    model_package_arn = next((p['ModelPackageArn'] for p in list_packages('PendingManualApproval') if is_deployable(p['ModelPackageArn'])), None)

if model_package_arn is None:
    raise RuntimeError(
        f'No deployable Approved or PendingManualApproval model version in {model_package_group}. '
        'Run the pipeline in notebook 03 first.'
    )

described = sm_client.describe_model_package(ModelPackageName=model_package_arn)
metadata = described.get('CustomerMetadataProperties', {})
print(f'Selected model package: {model_package_arn}')
print(f'  approval status:      {described.get("ModelApprovalStatus")}')
print(f'  image:                {described["InferenceSpecification"]["Containers"][0]["Image"]}')
print(f'  MLflow model id:      {metadata.get("MlflowModelId", "n/a")}')

#### Approve the model version
If the selected version has the status `PendingManualApproval`, the next cell approves it. In a real MLOps setup a person or an automated quality gate approves the version, and the approval starts a deployment pipeline. You build this pipeline in step 5.

<div class="alert alert-warning">
If you already created the model deployment project from notebook 05 (for example on a re-run of the workshop), approving a model version also triggers its EventBridge rule, which starts the model deployment CI/CD pipeline.
</div>

In [ ]:
model_package = ModelPackage.get(model_package_name=model_package_arn)
# DescribeModelPackage does not return ModelPackageName for versioned packages,
# which breaks refresh()/update(). The API accepts an ARN in that field.
model_package.model_package_name = model_package.model_package_arn

if model_package.model_approval_status != 'Approved':
    model_package.update(model_approval_status='Approved')
    model_package.refresh()

assert model_package.model_approval_status == 'Approved', model_package.model_approval_status
print(f'Approval status: {model_package.model_approval_status}')

#### Validate the inference code in the notebook
Before you deploy, check that the model and its serving code work on the test data, without an endpoint and without cost. The next cell downloads the same MLflow artifacts that the endpoint downloads, imports the logged `code/inference.py`, and calls its handlers. It also checks that the test data has the number of features that the model expects.

In [ ]:
model_id = metadata.get('MlflowModelId')
if not model_id:
    print('[WARN] Model package has no MlflowModelId metadata, skipping local validation.')
else:
    local_model_dir = mlflow.artifacts.download_artifacts(artifact_uri=f'models:/{model_id}')
    spec = importlib.util.spec_from_file_location(
        'user_inference_module', os.path.join(local_model_dir, 'code', 'inference.py')
    )
    inference_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(inference_module)
    local_model = inference_module.model_fn(local_model_dir)

    assert local_model.num_features() == X_test.shape[1], (
        f'Model expects {local_model.num_features()} features, test data has {X_test.shape[1]}'
    )

    payload = '\n'.join(','.join(map(str, row)) for row in X_test.values).encode('utf-8')
    data = inference_module.input_fn(payload, 'text/csv')
    body, _ = inference_module.output_fn(inference_module.predict_fn(data, local_model), 'text/csv')
    local_predictions = np.array(body.strip().split('\n'), dtype=float)
    print(f'Local test AUC: {roc_auc_score(y_test, local_predictions):.4f}')

### Deploy the model to a real-time endpoint with data capture
A real-time endpoint serves predictions with low latency over HTTPS. Data capture saves the requests to the endpoint and the predictions of the model in S3. The monitoring in part 2 uses this data.

#### Deploy the model package
You don't need the image, the script, or the model data location to deploy from the registry. The `InferenceSpecification` of the model package has them, so a `ContainerDefinition` with the model package ARN is enough. The three resources map one to one to the SageMaker API: a `Model` (what to serve), an `EndpointConfig` (the instances and the **data capture** settings), and an `Endpoint` (the running HTTPS service).

Data capture can save the inputs, the outputs, or both, for all or a sample of the requests. This endpoint captures all inputs and outputs.

<div class="alert alert-info">
The endpoint has no authentication of its own. IAM controls the access with the permission <code>sagemaker:InvokeEndpoint</code>, and the endpoint isn't open to the public internet.
</div>

In [ ]:
inference_instance_type = 'ml.m5.xlarge'
endpoint_name = name_from_base("from-idea-to-prod-monitor-ep-", short=True)
model_name = endpoint_name
capture_s3_uri = f's3://{bucket_name}/{bucket_prefix}/data-capture'

Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(model_package_name=model_package_arn),
    execution_role_arn=sm_role,
)

EndpointConfig.create(
    endpoint_config_name=endpoint_name,
    production_variants=[
        ProductionVariant(
            variant_name='AllTraffic',
            model_name=model_name,
            initial_instance_count=1,
            instance_type=inference_instance_type,
        )
    ],
    data_capture_config=DataCaptureConfig(
        enable_capture=True,
        initial_sampling_percentage=100,
        destination_s3_uri=capture_s3_uri,
        capture_options=[CaptureOption(capture_mode='Input'), CaptureOption(capture_mode='Output')],
        capture_content_type_header=CaptureContentTypeHeader(csv_content_types=['text/csv']),
    ),
)

Endpoint.create(endpoint_name=endpoint_name, endpoint_config_name=endpoint_name)

print(f'Creating endpoint {endpoint_name} from {model_package_arn}')
print(f'Data capture destination: {capture_s3_uri}')
print('Poll its status in the next cell (this takes a few minutes).')

In [ ]:
# Endpoint.create() returns as soon as the request is accepted, so poll until the
# endpoint reaches a terminal state.

sagemaker_client = boto3.client('sagemaker')
print(f'Polling endpoint: {endpoint_name}')

terminal_states = {'InService', 'Failed'}
while True:
    desc = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
    status = desc['EndpointStatus']
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != 'InService':
    raise RuntimeError(
        f"Endpoint {endpoint_name} deployment ended with status '{status}': "
        f"{desc.get('FailureReason')}"
    )
print(f'Endpoint {endpoint_name} is InService.')

In [ ]:
display(HTML(
    f'<b>See the SageMaker <a target="_blank" rel="noopener noreferrer" href="https://studio-{domain_id}.studio.{region}.sagemaker.aws/'
    f'inference-experience/endpoints/details/{endpoint_name}">inference endpoint</a></b>'
))

#### Explore the lineage of the model version
The lineage view connects the deployed model version with the pipeline execution, the training job, and the datasets that produced it.

In [ ]:
display(HTML(
    f'<b>See <a target="_blank" rel="noopener noreferrer" href="https://studio-{domain_id}.studio.{region}.sagemaker.aws/registry/registered-models/'
    f'{model_package.model_package_group_name}/versions/version-{model_package.model_package_version}/lineage">'
    'the model package version lineage</a> in the Studio UI</b>'
))

![](img/model-version-lineage.png)

### Send traffic to the endpoint
To evaluate the model quality, you compare the predictions with the ground truth labels. In production the labels often arrive later. For a sales forecast for next month, the actual sales are known only after the month. For a product recommendation, the purchase decision may be known within minutes.

This notebook uses the test dataset as production traffic, so the labels are known. To mimic production, the next cell saves the labels separately, to merge them with the captured predictions later. Each request gets a unique inference ID, such as `inf-000010`, for the merge.

<div class="alert alert-warning"><b>Simulated drift:</b> to demonstrate the drift detection, the next cell shifts two features of the test data before you send it to the endpoint: more contacts per campaign and older clients. Never change production data like this.</div>

In [ ]:
test_features = X_test.copy()
rng = np.random.default_rng(42)

# SIMULATED PRODUCTION DRIFT: apply shifts here so the Data Capture actually
# contains drifted records (the Processing Job compares capture vs baseline).
# Do NOT do this in production — this is purely to demonstrate drift detection.

# 1. More contacts per campaign: `campaign` is min-max scaled to [0, 1]
test_features['campaign'] = (
    test_features['campaign'] * 1.5 + rng.normal(0, 0.02, len(test_features))
).clip(0, 1)

# 2. An older customer base: move 70% of the records up one age bucket
age_cols = [c for c in test_features.columns if c.startswith('age_')]
bucket = test_features[age_cols].values.argmax(axis=1)
bucket = np.minimum(bucket + (rng.random(len(test_features)) < 0.7), len(age_cols) - 1)
test_features[age_cols] = np.eye(len(age_cols), dtype=int)[bucket]

ground_truth = {f'inf-{idx:06d}': int(label) for idx, label in enumerate(y_test)}
gt_df = pd.DataFrame(list(ground_truth.items()), columns=['inference_id', 'target'])
gt_df.to_csv('data/ground_truth.csv', index=False)
print(f'Ground truth stored for {len(gt_df)} records')

Send the test data, without the labels, to the endpoint.

In [ ]:
runtime_client = boto3.client('sagemaker-runtime', region_name=region)

endpoint_predictions = []
print(f'Sending {len(test_features)} requests to endpoint...')
for idx, (i, row) in enumerate(test_features.iterrows()):
    inference_id = f'inf-{idx:06d}'
    payload = ','.join(str(v) for v in row.values)
    response = runtime_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='text/csv',
        Accept='text/csv',
        Body=payload,
        InferenceId=inference_id
    )
    endpoint_predictions.append(float(response['Body'].read().decode('utf-8').strip()))

print(f'Sent {len(test_features)} requests')

Compare the predictions of the endpoint with the labels. The traffic contains the simulated drift, so compare this AUC with the AUC in the notebook before the deployment.

In [ ]:
endpoint_predictions = np.array(endpoint_predictions)
print(f'Endpoint test AUC (drifted traffic): {roc_auc_score(y_test, endpoint_predictions):.4f}')

pd.crosstab(
    index=y_test.values,
    columns=np.round(endpoint_predictions),
    rownames=['actuals'],
    colnames=['predictions'],
)

#### Read the captured data
The captured data arrives in S3 after 1-2 minutes. Data capture stops capturing if the disk usage of the endpoint instance is high, so that it doesn't affect the inference requests.

In [ ]:
s3_client = boto3.client('s3')
capture_prefix = f'{bucket_prefix}/data-capture/{endpoint_name}/AllTraffic'

print('Waiting for data capture files in S3 (may take 1-2 minutes)...')
for attempt in range(12):
    resp = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=capture_prefix, MaxKeys=1)
    if resp.get('KeyCount', 0) > 0:
        print(f'Data capture files found after {(attempt+1)*15}s')
        break
    time.sleep(15)
else:
    print('No capture files found yet. They may still be buffering.')

# List captured files
resp = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=capture_prefix)
capture_keys = [obj['Key'] for obj in resp.get('Contents', [])]
print(f'\nFound {len(capture_keys)} capture file(s)')

Data capture stores the inputs and outputs in [JSON Lines files](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-data-capture-endpoint.html). The next cell loads them into a pandas DataFrame.

In [ ]:
import base64

captured_records = []

for key in capture_keys:
    obj = s3_client.get_object(Bucket=bucket_name, Key=key)
    for line in obj['Body'].read().decode('utf-8').strip().split('\n'):
        record = json.loads(line)
        inference_id = record.get('eventMetadata', {}).get('inferenceId', '')

        input_data = record.get('captureData', {}).get('endpointInput', {})
        output_data = record.get('captureData', {}).get('endpointOutput', {})

        raw_input = input_data.get('data', '')
        if input_data.get('encoding') == 'BASE64':
            raw_input = base64.b64decode(raw_input).decode('utf-8')

        raw_output = output_data.get('data', '')
        if output_data.get('encoding') == 'BASE64':
            raw_output = base64.b64decode(raw_output).decode('utf-8')

        captured_records.append({
            'inference_id': inference_id,
            'input': raw_input.strip(),
            'prediction_proba': float(raw_output.strip()) if raw_output.strip() else None
        })

captured_df = pd.DataFrame(captured_records)
captured_df['prediction'] = (captured_df['prediction_proba'] > 0.5).astype(int)

feature_cols = pd.DataFrame(
    captured_df['input'].str.split(',').tolist(),
    columns=test_features.columns
).astype(float)
captured_df = pd.concat([captured_df[['inference_id', 'prediction_proba', 'prediction']], feature_cols], axis=1)

print(f'Parsed {len(captured_df)} captured records with inferenceId')
print(captured_df[['inference_id', 'prediction_proba', 'prediction']].head())

---

## Part 2: Monitor the model

### Detect data drift with Evidently
Data drift is a change of the statistical properties of the input features over time. A model trained on the old distribution can lose accuracy on the new one. Drift detection warns you before the business notices, and tells you when to retrain the model.

[Evidently](https://docs.evidentlyai.com/) compares a current dataset with a reference dataset. It chooses a statistical test for each column, such as the Kolmogorov-Smirnov test or the chi-squared test, and creates interactive HTML reports.

Load the baseline dataset as the reference. It has the distribution of the training data.

In [ ]:
reference_data = pd.read_csv('data/baseline.csv')

print(f'Reference (baseline) data shape: {reference_data.shape}')
print('\nReference data represents the training data distribution')
print('\nReference data statistics:')
print(reference_data.describe())

The captured inputs are the current dataset. They contain the drift that you added before you sent the traffic.

In [ ]:
feature_columns = [c for c in captured_df.columns if c not in ['inference_id', 'prediction_proba', 'prediction']]
production_data = captured_df[feature_columns].copy()

print(f'Production data shape: {production_data.shape}')
print('\nProduction data statistics:')
print(production_data.describe())


Evidently has [presets for data drift](https://docs.evidentlyai.com/metrics/preset_data_drift) that you can customize for your use case. Besides the full report, log selected drift values as MLflow metrics, to compare runs and to create alerts. The next helper function extracts these values from the Evidently results.

In [ ]:
def log_metrics_from_dict(d):
    metrics = d.get("metrics", []) or []

    drifted_columns_logged = False

    for m in metrics:
        metric_name = m.get("metric_name", "")
        cfg = m.get("config", {}) or {}
        val = m.get("value")

        # 1) DriftedColumnsCount: always log
        if metric_name.startswith("DriftedColumnsCount"):
            drifted_columns_logged = True
            # value is a dict: {"count": ..., "share": ...}
            if isinstance(val, dict):
                count = float(val.get("count", 0.0))
                share = float(val.get("share", 0.0))
            else:
                # Fallback if structure changes
                count = float(val) if val is not None else 0.0
                share = 0.0

            mlflow.log_metric("DriftedColumnsCount.count", count)
            mlflow.log_metric("DriftedColumnsCount.share", share)
            continue

        # 2) ValueDrift: log only if value > threshold
        if metric_name.startswith("ValueDrift"):
            threshold = cfg.get("threshold")
            column = cfg.get("column")

            if threshold is None or column is None:
                continue

            try:
                numeric_val = float(val)
            except (TypeError, ValueError):
                continue

            if numeric_val > float(threshold):
                key = f"ValueDrift:{column}"
                mlflow.log_metric(key, numeric_val)

    # 3) If there was no DriftedColumnsCount metric at all, log 0
    if not drifted_columns_logged:
        mlflow.log_metric("DriftedColumnsCount", 0.0)

Create a local folder for the reports.

In [ ]:
os.makedirs('reports', exist_ok=True)

The next cell runs the `DataDriftPreset` and the `DataSummaryPreset` on the baseline and the captured data. It logs the drift metrics and the HTML and JSON reports in MLflow.

In [ ]:
mlflow.set_experiment(mlflow_experiment_name)
timestamp_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')

with mlflow.start_run(run_name=f"data_drift_quality_monitoring_{timestamp_suffix}") as drift_run:
    drift_run_id = drift_run.info.run_id
    
    mlflow.log_params({
        'reference_data_size': len(reference_data),
        'current_data_size': len(production_data),
        'monitoring_timestamp': datetime.now().isoformat(),
        'model_name': model_name,
        'model_package_arn': model_package_arn,
    })

    # Wrap DataFrames in Evidently Dataset objects
    drift_data_definition = DataDefinition()
    reference_dataset = Dataset.from_pandas(reference_data, data_definition=drift_data_definition)
    production_dataset = Dataset.from_pandas(production_data, data_definition=drift_data_definition)
    
    print('Creating Evidently data drift report...')
    data_drift_report = Report(metrics=[
        DataDriftPreset(),
    ])
    data_drift_report_snapshot = data_drift_report.run(
        reference_data=reference_dataset,
        current_data=production_dataset
    )
    
    data_drift_report_filename = f'data_drift_report_{timestamp_suffix}'
    data_drift_report_snapshot.save_html(f'reports/{data_drift_report_filename}.html')
    data_drift_report_snapshot.save_json(f'reports/{data_drift_report_filename}.json')
    
    mlflow.log_artifact(f'reports/{data_drift_report_filename}.html', 'evidently_report_data_drift_html')
    mlflow.log_artifact(f'reports/{data_drift_report_filename}.json', 'evidently_report_data_drift_json')
    
    drift_results = data_drift_report_snapshot.dict()
    log_metrics_from_dict(drift_results)

    print('Creating Evidently data summary report...')
    data_quality_report = Report(metrics=[
        DataSummaryPreset(),
    ])
    data_quality_report_snapshot = data_quality_report.run(
        reference_data=reference_data,
        current_data=production_data
    )
    
    data_quality_filename_html = f'data_quality_report_{timestamp_suffix}.html'
    data_quality_report_snapshot.save_html(f'reports/{data_quality_filename_html}')
    print(f'Data quality report saved to: {data_quality_filename_html}')
    mlflow.log_artifact(f'reports/{data_quality_filename_html}', 'evidently_report_data_quality')

Show the data drift report in the notebook. The `campaign` and `age_*` columns show the simulated drift.

In [ ]:
from IPython.display import IFrame

print('Displaying Evidently Data Drift Report:')
IFrame(src=f'reports/{data_drift_report_filename}.html', width=1000, height=800)

#### View the drift metrics in MLflow
Run the next cells and choose the links to open the monitoring experiment `from-idea-to-prod-monitoring` in the MLflow UI. Open the run `data_drift_quality_monitoring_...` to see:
- **Metrics**: the drift scores of the drifted features and the share of drifted columns
- **Artifacts**: the HTML and JSON reports
- **Parameters**: the dataset sizes, the model, and the time of the check

In [ ]:
from html import escape

presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn,
    ExpiresInSeconds=300,
    SessionExpirationDurationInSeconds=1800
)['AuthorizedUrl']
monitoring_runs_link = (
    f"{presigned_url.split('/auth')[0]}/#/experiments/"
    f"{monitoring_experiment.experiment_id}/runs?workspace=default"
)

# Open the MLflow UI first: the presigned URL authenticates your browser session
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(presigned_url, quote=True)}">'
    '1. Open the MLflow UI and authenticate</a></b>'
))

In [ ]:
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(monitoring_runs_link, quote=True)}">'
    '2. Open the monitoring runs</a></b>'
))

### Evaluate the model quality with Evidently
The model quality compares the predictions with the ground truth labels. For a binary classifier, the main metrics are accuracy, precision, recall, F1 score, and ROC AUC, and the confusion matrix. For the bank campaign, high precision means fewer calls to clients who won't subscribe, and high recall means fewer missed subscribers. You choose the decision threshold based on the cost of a call and the value of a subscription.

The Evidently `ClassificationPreset` calculates these metrics and the quality per class.

Merge the captured predictions with the saved labels by the inference ID.

In [ ]:
gt_df = pd.read_csv('data/ground_truth.csv')
classification_eval_data = captured_df.merge(gt_df, on='inference_id', how='inner')

print(f'Merged dataset shape: {classification_eval_data.shape}')
print(f'  Captured records: {len(captured_df)}')
print(f'  Ground truth records: {len(gt_df)}')
print(f'  Matched records: {len(classification_eval_data)}')
print(f'\nActual labels distribution:')
print(classification_eval_data['target'].value_counts())
print(f'\nPredicted labels distribution:')
print(classification_eval_data['prediction'].value_counts())

Evidently has [presets for model quality](https://docs.evidentlyai.com/metrics/preset_classification). The next helper function logs the classification metrics in MLflow.

In [ ]:
def log_evidently_classification_metrics_to_mlflow(metrics_dict):
    """Log Evidently classification metrics to MLflow using only metric prefix"""
    for metric in metrics_dict.get('metrics', []):
        # Extract only prefix before first '('
        metric_name_full = metric.get('metric_name', '')
        metric_name = metric_name_full.split('(')[0].strip()
        
        value = metric.get('value')
        
        if isinstance(value, dict):
            # Handle dict values like F1ByLabel, PrecisionByLabel, etc.
            for label, val in value.items():
                mlflow.log_metric(f"{metric_name}_label_{label}", float(val))
        else:
            # Handle scalar values (including np.float64)
            mlflow.log_metric(metric_name, float(value))

The next cell runs the `ClassificationPreset` on the merged data and logs the metrics and the HTML and JSON reports in MLflow.

In [ ]:
with mlflow.start_run(run_name=f'model_quality_{timestamp_suffix}') as run:
    eval_data = classification_eval_data.copy()
    print(f"Evaluation dataset columns: {eval_data.columns.tolist()}")
    print(f"Evaluation dataset shape: {eval_data.shape}")
    
    # Create DataDefinition with BinaryClassification to tell Evidently which columns to use
    # This is REQUIRED for ClassificationPreset to work
    data_definition = DataDefinition(
        classification=[
            BinaryClassification(
                target='target',              # Column with actual labels
                prediction_labels='prediction', # Column with predicted labels
                pos_label=1,                  # Positive class label
            )
        ],
    )
    
    # Wrap the DataFrame in an Evidently Dataset
    eval_dataset = Dataset.from_pandas(
        eval_data,
        data_definition=data_definition,
    )
    
    print('\nCreating Evidently classification performance report...')
    classification_report = Report(metrics=[
        ClassificationPreset(),
    ])
    classification_report_snapshot = classification_report.run(
        reference_data=None,
        current_data=eval_dataset,
    )
    
    classification_report_filename = f'classification_report_{timestamp_suffix}'
    classification_report_snapshot.save_html(f'data/{classification_report_filename}.html')
    print(f'Classification report saved to: data/{classification_report_filename}.html')
    
    classification_report_snapshot.save_json(f'data/{classification_report_filename}.json')
    print(f'Classification report saved to: data/{classification_report_filename}.json')
    
    mlflow.log_artifact(f'data/{classification_report_filename}.html', 'evidently_classification_report_html')
    mlflow.log_artifact(f'data/{classification_report_filename}.json', 'evidently_classification_report_json')
    
    classification_report_dict = classification_report_snapshot.dict()
    log_evidently_classification_metrics_to_mlflow(classification_report_dict)

#### Interpret the model quality
Open the classification report in the MLflow run `model_quality_...`. Compare its metrics with the test AUC of the pipeline in step 3 and with the AUC in the notebook before the deployment. A drop shows how much the simulated drift hurts the model. In production you would retrain the model on recent data when the quality falls below a threshold.

---

### Automate the monitoring in a scheduled pipeline

The calculations in the notebook are good for an analysis on demand. In production the drift and quality checks run on a schedule, without a notebook. The next cells run the Evidently code in a **SageMaker processing job** in a **SageMaker pipeline** that:

- Pulls the latest Data Capture files from the endpoint and compares them to the training baseline
- Logs the Evidently reports and drift metrics to MLflow
- Publishes an **SNS** notification when the share of drifted features exceeds a threshold, so a human (or a downstream retraining pipeline) can react

<div class="alert alert-info" role="alert">
<strong>Heads up:</strong> the pipeline execution role needs <code>sns:Publish</code> on the created topic to send alerts, and <code>scheduler:CreateSchedule</code> / <code>iam:PassRole</code> if you enable the EventBridge schedule. Workshop accounts may not have these permissions. In this case the code logs a warning and the job still succeeds.
</div>

#### Create the SNS topic for drift alerts

The next cell creates an SNS topic and subscribes your email address, if you set `ALERT_EMAIL`. The email owner must confirm the subscription. If the execution role doesn't have SNS permissions, the cell logs a warning. The pipeline still runs, but doesn't publish alerts.

In [ ]:
ALERT_EMAIL = ''  # optional: set to receive drift alerts by email (must be confirmed)
SNS_TOPIC_NAME = 'sagemaker-evidently-drift-alerts'

sns_topic_arn = ''
try:
    sns_client = boto3.client('sns', region_name=region)
    sns_topic_arn = sns_client.create_topic(Name=SNS_TOPIC_NAME)['TopicArn']
    print(f'SNS topic ready: {sns_topic_arn}')
    if ALERT_EMAIL:
        sns_client.subscribe(TopicArn=sns_topic_arn, Protocol='email', Endpoint=ALERT_EMAIL)
        print(f'Subscription requested for {ALERT_EMAIL} — confirm via email to activate.')
except Exception as e:
    print(f'[WARN] Could not set up SNS ({e}). Pipeline will still run without alerts.')


#### The processing job script
The script [`scripts/evidently_monitor.py`](scripts/evidently_monitor.py) runs the same Evidently checks as the previous sections. It reads the baseline, the captured data, and the ground truth from the processing job inputs, logs the reports in MLflow, and publishes an SNS alert if a check fails. Expand the panel to see the script.

In [ ]:
monitor_script = open("scripts/evidently_monitor.py").read()
display(HTML(
    '<details><summary><b>View source: scripts/evidently_monitor.py</b></summary>'
    f'<pre style="max-height: 32rem; overflow: auto;"><code>{escape(monitor_script)}</code></pre></details>'
))

#### Define the monitoring pipeline

The pipeline has one `ProcessingStep` that runs `scripts/evidently_monitor.py` with a `FrameworkProcessor`. Pipeline parameters change each execution without a new pipeline definition:

- `DriftThreshold`: the share of drifted columns that triggers an alert
- `LookbackDays`: use only the captured records of the last N days. `0` uses all records.
- `CriticalFeatures`: comma-separated features that trigger an alert if they drift, even if the share of drifted columns is below the threshold
- `MinF1`: if greater than 0, also alert if the F1 score is below this value. Needs the ground truth.
- `SnsTopicArn`, `EndpointName`: the SNS topic and the endpoint that you created

The processing job uses the SageMaker AI scikit-learn image, which doesn't include Evidently and MLflow. `FrameworkProcessor` installs the packages in [`scripts/requirements.txt`](scripts/requirements.txt) before it runs the script. The file pins the Evidently version that was tested with this image.

In [ ]:
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, CacheConfig
from sagemaker.core.workflow.parameters import ParameterFloat, ParameterString, ParameterInteger
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.processing import FrameworkProcessor
from sagemaker.core.shapes import ProcessingInput, ProcessingS3Input, ProcessingOutput, ProcessingS3Output
from sagemaker.core import image_uris

monitoring_pipeline_name = 'evidently-monitoring-pipeline'

p_drift_threshold = ParameterFloat(name='DriftThreshold', default_value=0.05)
p_sns_topic = ParameterString(name='SnsTopicArn', default_value=sns_topic_arn or '')
p_endpoint = ParameterString(name='EndpointName', default_value=endpoint_name)
p_lookback_days = ParameterInteger(name='LookbackDays', default_value=7)
p_critical_features = ParameterString(name='CriticalFeatures', default_value='campaign,age_18-29')
p_min_f1 = ParameterFloat(name='MinF1', default_value=0.0)

endpoint_capture_s3 = f'{capture_s3_uri}/{endpoint_name}/AllTraffic'

ground_truth_s3 = sagemaker_session.upload_data(
    'data/ground_truth.csv', bucket_name, f'{bucket_prefix}/data/ground-truth'
)

processor_image_uri = image_uris.retrieve(framework='sklearn', version='1.4-2', region=region)

monitor_processor = FrameworkProcessor(
    image_uri=processor_image_uri,
    role=sm_role,
    instance_type='ml.m5.large',
    instance_count=1,
    sagemaker_session=PipelineSession(),
    base_job_name='evidently-monitor',
    command=['python3'],
)

monitor_inputs = [
    ProcessingInput(input_name='baseline', s3_input=ProcessingS3Input(
        s3_uri=baseline_s3, local_path='/opt/ml/processing/input/baseline', s3_data_type='S3Prefix')),
    ProcessingInput(input_name='capture', s3_input=ProcessingS3Input(
        s3_uri=endpoint_capture_s3, local_path='/opt/ml/processing/input/capture', s3_data_type='S3Prefix')),
    ProcessingInput(input_name='ground_truth', s3_input=ProcessingS3Input(
        s3_uri=ground_truth_s3, local_path='/opt/ml/processing/input/ground_truth', s3_data_type='S3Prefix')),
]

monitor_outputs = [
    ProcessingOutput(output_name='reports', s3_output=ProcessingS3Output(
        s3_uri=f's3://{bucket_name}/{bucket_prefix}/evidently-reports',
        local_path='/opt/ml/processing/output',
        s3_upload_mode='EndOfJob')),
]

monitor_step = ProcessingStep(
    name='EvidentlyMonitoring',
    step_args=monitor_processor.run(
        code='evidently_monitor.py',
        source_dir='scripts',
        inputs=monitor_inputs,
        outputs=monitor_outputs,
        arguments=[
            '--mlflow-tracking-uri', mlflow_arn,
            '--mlflow-experiment', mlflow_experiment_name,
            '--drift-threshold', p_drift_threshold.to_string(),
            '--lookback-days', p_lookback_days.to_string(),
            '--critical-features', p_critical_features,
            '--min-f1', p_min_f1.to_string(),
            '--sns-topic-arn', p_sns_topic,
            '--endpoint-name', p_endpoint,
        ],
    ),
    cache_config=CacheConfig(enable_caching=False),
)

monitoring_pipeline = Pipeline(
    name=monitoring_pipeline_name,
    parameters=[p_drift_threshold, p_sns_topic, p_endpoint, p_lookback_days, p_critical_features, p_min_f1],
    steps=[monitor_step],
)
print(f'Pipeline defined: {monitoring_pipeline_name}')

#### Create the pipeline and an optional schedule
`pipeline.upsert()` creates the pipeline or updates its definition. [Amazon EventBridge Scheduler](https://docs.aws.amazon.com/scheduler/latest/UserGuide/what-is-scheduler.html) can start the pipeline on a schedule. Set `ENABLE_SCHEDULE = True` to create a daily run at 08:00 UTC. The schedule is off by default, because each run starts a processing job.

In [ ]:
ENABLE_SCHEDULE = False  # set to True to create the daily run
SCHEDULE_EXPRESSION = 'cron(0 8 * * ? *)'  # every day at 08:00 UTC
# SCHEDULE_EXPRESSION = 'cron(0/20 * * * ? *)' # every 20 minutes
SCHEDULE_NAME = f'{monitoring_pipeline_name}-daily'
# SCHEDULE_NAME = f'{monitoring_pipeline_name}-20mins'

monitoring_pipeline.upsert(role_arn=sm_role)
print(f'Pipeline upserted: {monitoring_pipeline.name}')

if ENABLE_SCHEDULE:
    try:
        scheduler = boto3.client('scheduler', region_name=region)
        pipeline_arn = f'arn:aws:sagemaker:{region}:{boto3.client("sts").get_caller_identity()["Account"]}:pipeline/{monitoring_pipeline.name}'
        scheduler.create_schedule(
            Name=SCHEDULE_NAME,
            ScheduleExpression=SCHEDULE_EXPRESSION,
            FlexibleTimeWindow={'Mode': 'OFF'},
            Target={
                'Arn': pipeline_arn,
                'RoleArn': sm_role,
                'SageMakerPipelineParameters': {
                    'PipelineParameterList': [
                        {'Name': 'DriftThreshold', 'Value': '0.05'},
                        {'Name': 'LookbackDays', 'Value': '7'},
                        {'Name': 'CriticalFeatures', 'Value': 'campaign,age_18-29'},
                        {'Name': 'MinF1', 'Value': '0.0'},
                        {'Name': 'SnsTopicArn', 'Value': sns_topic_arn or ''},
                        {'Name': 'EndpointName', 'Value': endpoint_name},
                    ]
                },
            }
        )
        print(f'Schedule created: {SCHEDULE_NAME} ({SCHEDULE_EXPRESSION})')
    except Exception as e:
        print(f'[WARN] Could not create schedule ({e}). Start the pipeline manually below.')
else:
    print('Schedule disabled (ENABLE_SCHEDULE=False). Start the pipeline manually below.')

#### Run the pipeline
Start an execution to test the pipeline. The next cell starts the pipeline, shows a link to the execution graph in Studio, waits until the execution completes, and lists the steps. The cell after it shows the MLflow run of the job.

<div class="alert alert-info">Data capture can take 1-2 minutes to write the captured data to S3. If the job logs <code>No capture records found</code>, wait a minute and run the cell again.</div>

In [ ]:
execution = monitoring_pipeline.start(
    parameters={
        'DriftThreshold': 0.05,
        'LookbackDays': 7,              # compare only capture from the last 7 days; set 0 to disable
        'CriticalFeatures': 'campaign,age_18-29',  # alert individually if any of these drift
        'MinF1': 0.0,                   # set > 0 to also alert on F1 degradation
        'SnsTopicArn': sns_topic_arn or '',
        'EndpointName': endpoint_name,
    }
)
execution_id = execution.describe()['PipelineExecutionArn'].split('/')[-1]
print(f'Pipeline execution started: {execution_id}')

display(HTML(
    f'<b>See <a target="_blank" rel="noopener noreferrer" href="https://studio-{domain_id}.studio.{region}.sagemaker.aws/'
    f'pipelines/{monitoring_pipeline_name}/executions/{execution_id}/graph">the pipeline execution</a> in the Studio UI</b>'
))

%time execution.wait()
execution.list_steps()

In [ ]:
# Show the MLflow run produced by the scheduled monitoring job
from mlflow import MlflowClient

mlflow.set_tracking_uri(mlflow_arn)
client = MlflowClient()
exp = client.get_experiment_by_name(mlflow_experiment_name)
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="tags.mlflow.runName LIKE 'scheduled_monitoring_%'",
    order_by=['start_time DESC'],
    max_results=1,
)
if runs:
    r = runs[0]
    print(f"MLflow run: {r.info.run_name}")
    print(f"  run_id:    {r.info.run_id}")
    print(f"  share:     {r.data.metrics.get('DriftedColumnsCount.share', 'n/a')}")
else:
    print('No monitoring run found yet — check the Studio pipeline execution logs.')


---

## Summary
In this notebook you:
- selected the model version from the pipeline in the SageMaker Model Registry, approved it, and validated its serving code in the notebook
- deployed the model package to a real-time endpoint with data capture
- sent test traffic with simulated drift to the endpoint
- detected the data drift and evaluated the model quality on the captured data with Evidently, and logged the reports and metrics in MLflow
- ran the same checks in a processing job in a SageMaker pipeline, with an optional daily schedule and SNS alerts

## Clean up
The endpoint costs money while it runs. Run the next cell to delete the endpoint, the schedule, the monitoring pipeline, and the SNS topic of this notebook, unless you continue with the optional notebook [additional-topics/04-monitoring-sagemaker-native](additional-topics/04-monitoring-sagemaker-native.ipynb). Notebook 05 creates its own endpoints.

In [ ]:
sm_client = boto3.client('sagemaker')

try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f'Deleting endpoint: {endpoint_name} (this may take a minute)')
except Exception as e:
    print(f'Error deleting endpoint: {e}')

try:
    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print(f'Deleted endpoint config')
except Exception as e:
    print(f'Error deleting endpoint config: {e}')

try:
    sm_client.delete_model(ModelName=model_name)
    print(f'Deleted model: {model_name}')
except Exception as e:
    print(f'Error deleting model: {e}')

try:
    boto3.client('scheduler', region_name=region).delete_schedule(Name=SCHEDULE_NAME)
    print(f'Deleted schedule: {SCHEDULE_NAME}')
except Exception as e:
    print(f'No schedule to delete or error: {e}')

try:
    monitoring_pipeline.delete()
    print(f'Deleted pipeline: {monitoring_pipeline.name}')
except Exception as e:
    print(f'Error deleting pipeline: {e}')

try:
    if sns_topic_arn:
        sns_client.delete_topic(TopicArn=sns_topic_arn)
        print(f'Deleted SNS topic: {sns_topic_arn}')
except Exception as e:
    print(f'Error deleting SNS topic: {e}')

try:
    reports_prefix = f'{bucket_prefix}/evidently-reports'
    objects = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=reports_prefix).get('Contents', [])
    if objects:
        s3_client.delete_objects(Bucket=bucket_name, Delete={'Objects': [{'Key': o['Key']} for o in objects]})
        print(f'Deleted {len(objects)} Evidently report objects from s3://{bucket_name}/{reports_prefix}')
except Exception as e:
    print(f'Error deleting Evidently reports: {e}')

print('\nThe MLflow runs and the S3 data stay for later analysis.')

## Continue with step 5
Open the step 5 notebook [05-templatize-deployment](05-templatize-deployment.ipynb). In step 5 you automate the model deployment with a CI/CD pipeline in a SageMaker project.

## Further development ideas for your real-world projects
- Start the monitoring pipeline when new captured data arrives in S3, with an [Amazon EventBridge rule](https://docs.aws.amazon.com/sagemaker/latest/dg/pipeline-eventbridge.html)
- Start a retraining pipeline when the drift or the F1 score crosses a threshold, and deploy the new model version after its approval
- Track the drift of the predictions and business KPIs, such as the subscription rate
- Build trend dashboards on the MLflow metrics
- Alert on failed pipeline executions, not only on drift
- Compare with the native [SageMaker Model Monitor](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor.html) in the optional notebook [additional-topics/04-monitoring-sagemaker-native](additional-topics/04-monitoring-sagemaker-native.ipynb)

---

## Additional resources

### Documentation
- [SageMaker Python SDK v3](https://sagemaker.readthedocs.io/en/stable/)
- [SageMaker AI MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- [Evidently Documentation](https://docs.evidentlyai.com/)

### GitHub repositories
- [Evidently GitHub](https://github.com/evidentlyai/evidently)
- [SageMaker Examples](https://github.com/aws/amazon-sagemaker-examples)

## Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>